# Arbitrage Calculator: Cloud GPU ML Matcher (v4-stable)
### bge-m3 + bge-reranker-v2-m3 | pure torch on T4 | WebSocket pipeline

### Instructions
1. **Runtime > Change runtime type** -> ensure **T4 GPU** is selected
2. **Runtime > Run all** (or Ctrl+F9). Auto-execute will trigger as soon as the runtime connects.

### What changed from v3
- Bi-encoder upgraded: MiniLM -> **BAAI/bge-m3** (state of the art for semantic retrieval)
- Reranker upgraded: NLI DeBERTa -> **BAAI/bge-reranker-v2-m3** (purpose-built reranker)
- Binary-only + sports-noise pre-filter drops unmatchable markets before encoding
- Date / number / proper-noun compatibility filters reject obvious mismatches

### What is intentionally NOT here
- No `faiss-gpu` (uses pure torch `util.cos_sim` like v3 - bulletproof on T4)
- No `spacy` (regex-based proper-noun overlap covers the same ground without the install)


In [ ]:
# 1. Install + GPU check
!pip install -q sentence-transformers torch httpx websockets pydantic nest_asyncio \
             rapidfuzz spacy
!python -m spacy download en_core_web_sm -q

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import httpx, websockets, asyncio, json, re, time
from typing import List, Dict, Any, Set, Tuple
import nest_asyncio
from rapidfuzz import fuzz as _rfuzz
import spacy as _spacy
nest_asyncio.apply()

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name} ({_p.total_memory/1e9:.1f} GB)')


In [ ]:
# 2. Load models onto T4(s) — multi-GPU aware, 3-stage pipeline
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f'Device: {DEVICE} | GPUs: {NUM_GPUS}')
for _i in range(NUM_GPUS):
    _p = torch.cuda.get_device_properties(_i)
    print(f'  GPU {_i}: {_p.name} ({_p.total_memory/1e9:.1f} GB)')

# Stage-1 bi-encoder: BAAI/bge-m3 (state-of-the-art dense retrieval)
print('Loading bi-encoder   (BAAI/bge-m3)...')
bi_model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

# Multi-process pool for dual-T4 parallel encoding
_encode_devices = [f'cuda:{i}' for i in range(NUM_GPUS)] if NUM_GPUS > 1 else None
_mp_pool = bi_model.start_multi_process_pool(_encode_devices) if _encode_devices else None
if _mp_pool:
    print(f'  bi-encoder pool: {_encode_devices}')

def encode_titles(titles, batch_size=256):
    if _mp_pool:
        import numpy as _np
        embs = SentenceTransformer.encode_multi_process(
            titles, _mp_pool, batch_size=batch_size, normalize_embeddings=True)
        return torch.tensor(embs, device=DEVICE)
    return bi_model.encode(
        titles, convert_to_tensor=True, batch_size=batch_size,
        normalize_embeddings=True, show_progress_bar=True)

# Stage-2 reranker: bge-reranker-v2-m3 (fast, robust cross-encoder)
print('Loading reranker     (BAAI/bge-reranker-v2-m3)...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

# Stage-3 reranker: bge-reranker-v2-gemma — LLM-based (Gemma-2B backbone).
# MUCH better at semantic equivalence judgements than standard cross-encoders.
# Used only on the final top-500 candidates to keep latency reasonable.
print('Loading deep-reranker (BAAI/bge-reranker-v2-gemma) — 2B LLM, may take a moment...')
try:
    deep_reranker = CrossEncoder(
        'BAAI/bge-reranker-v2-gemma',
        device=DEVICE,
        trust_remote_code=True,
        max_length=512,
    )
    print('  deep-reranker loaded.')
except Exception as _e:
    print(f'  WARN: deep-reranker load failed ({_e}). Stage 3 disabled.')
    deep_reranker = None

# spaCy NER for entity cross-checking (catches Trump/Biden style mismatches)
try:
    _nlp = _spacy.load('en_core_web_sm', disable=['parser', 'tagger', 'lemmatizer'])
    def get_entities(text: str) -> set:
        return {ent.text.lower() for ent in _nlp(text).ents
                if ent.label_ in ('PERSON','ORG','GPE','EVENT','NORP','FAC')}
    print('spaCy NER ready.')
except Exception as _e:
    print(f'  WARN: spaCy load failed ({_e}). NER disabled.')
    def get_entities(text: str) -> set: return set()

print('All models loaded.')


In [ ]:
# 3. WebSocket fetch + binary/noise pre-filter
# The backend rewrites the placeholder line into `WS_URL = "wss://...your-ngrok..."` before upload.
WS_URL_PLACEHOLDER = "REPLACE_ME"
WS_URL = WS_URL_PLACEHOLDER

# ── Polite scan-status polling ─────────────────────────────────────────────
# We check whether all market scans (incl. both IBKR rounds) have finished.
# We NEVER forcibly terminate an in-progress scan — just wait for it.
import time as _time
_backend_http = WS_URL.replace('wss://', 'https://').replace('/ws', '')
print(f'[status-poll] backend = {_backend_http}')

_MAX_WAIT    = 7200   # 2 hours hard ceiling
_POLL_SEC    = 60     # check every 60 s
_waited      = 0

while _waited < _MAX_WAIT:
    try:
        with httpx.Client(timeout=15) as _c:
            _r = _c.get(f'{_backend_http}/api/scan-status')
        _st = _r.json()
        _status      = _st.get('status', 'unknown')
        _phase       = _st.get('phase', '')
        _ibkr_rounds = _st.get('ibkr_scan_rounds_done', 0)
        _total       = _st.get('total_markets', 0)
        _msg         = _st.get('message', '')[:80]
        print(f'  [{_waited:>5}s] {_status} | {_phase} | markets={_total:,} | IBKR_rounds={_ibkr_rounds} | {_msg}')

        # Proceed when IBKR has done both rounds AND scan is no longer active
        _scan_done = _status in ('complete', 'idle', 'waiting_for_cloud')
        if _ibkr_rounds >= 2 and _scan_done and _total > 0:
            print(f'  ✓ All scans complete ({_ibkr_rounds} IBKR rounds, {_total:,} markets). Fetching...')
            break
        # Fallback: proceed if scan is done with lots of markets but ibkr_rounds not tracked
        if _scan_done and _total > 10000 and _ibkr_rounds == 0:
            print(f'  ✓ Scan complete ({_total:,} markets, legacy). Fetching...')
            break
        # Error fallback — don't block forever
        if _status == 'error':
            print(f'  Backend scan errored. Proceeding with available data.')
            break
    except Exception as _e:
        print(f'  Status check error (backend may still be starting): {_e}')

    _time.sleep(_POLL_SEC)
    _waited += _POLL_SEC
else:
    print(f'  Polling timeout after {_MAX_WAIT}s — proceeding with whatever data is available.')

# ── End of polite polling; now fetch markets ───────────────────────────────
# 3. WebSocket fetch + binary/noise pre-filter
# The backend rewrites the placeholder line into `WS_URL = "wss://...your-ngrok..."` before upload.
WS_URL_PLACEHOLDER = "REPLACE_ME"
WS_URL = WS_URL_PLACEHOLDER

NOISE_PATTERNS = re.compile(
    r'exact score|\bmap \d+\b|\bround \d+\b|\bset \d+\b|\bhalf \d+\b'
    r'|over/under|o/u \d|\+/-|handicap|total kills|total goals'
    r'|first blood|first dragon|odd/even|both teams to score',
    re.IGNORECASE
)

def is_binary(m):
    if not m.get('isBinary', True): return False
    if m.get('outcomeCount', 2) > 2: return False
    return True

async def fetch_markets_via_websocket():
    print(f'Connecting to backend: {WS_URL}')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({'type': 'subscribe', 'channel': 'markets'}))
        response = await ws.recv()
        data = json.loads(response)
        if data.get('type') != 'markets_data':
            raise Exception(f'Unexpected response: {data}')
        return data.get('markets', [])

all_markets_raw = asyncio.get_event_loop().run_until_complete(fetch_markets_via_websocket())
print(f'Raw markets received: {len(all_markets_raw):,}')

seen, unique = set(), []
for m in all_markets_raw:
    key = m.get('id') or m.get('marketUrl')
    if key not in seen:
        seen.add(key); unique.append(m)
print(f'After dedup        : {len(unique):,}')

binary = [m for m in unique if is_binary(m)]
print(f'Binary only        : {len(binary):,} (dropped {len(unique)-len(binary):,} multi-outcome)')

clean = [m for m in binary if not NOISE_PATTERNS.search(m.get('title',''))]
print(f'After noise filter : {len(clean):,} (dropped {len(binary)-len(clean):,} sports/gaming noise)')

by_platform_preview = {}
for m in clean:
    by_platform_preview.setdefault(m['platform'], []).append(m)
for plat, mkts in by_platform_preview.items():
    print(f'  {plat:12} {len(mkts):,}')

all_markets = clean


In [ ]:
# 4. Regex-only compatibility filters (no spacy)
PROPER_NOUN_RE = re.compile(r"\b([A-Z][a-zA-Z]{2,})\b")
DATE_RE_LIST = [
    re.compile(r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b'),
    re.compile(r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b'),
    re.compile(r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(\d{4}))?\b', re.IGNORECASE),
]
NUMBER_RE = re.compile(r'\d+(?:\.\d+)?')
STOPWORDS = {'Will','The','Be','By','At','In','On','For','Of','To','And','Or','A','An','Is','Are','Was','Were','Yes','No'}

def extract_dates(text: str) -> Set[Tuple[int,int,int]]:
    out = set()
    for pat in DATE_RE_LIST:
        for m in pat.finditer(text):
            try:
                if pat is DATE_RE_LIST[0]:
                    mo, d, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                elif pat is DATE_RE_LIST[1]:
                    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                else:
                    months = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
                    mo = months[m.group(1).lower()]; d = int(m.group(2))
                    y = int(m.group(3)) if m.group(3) else 0
                    if y: out.add((y, mo, d))
            except Exception:
                pass
    return out

def extract_numbers(text: str) -> Set[float]:
    return {float(n) for n in NUMBER_RE.findall(text) if not (2020 <= float(n) <= 2030)}

def extract_proper_nouns(text: str) -> Set[str]:
    return {w for w in PROPER_NOUN_RE.findall(text) if w not in STOPWORDS}

def are_compatible(title_a: str, title_b: str) -> Tuple[bool, str]:
    da, db = extract_dates(title_a), extract_dates(title_b)
    if da and db and da.isdisjoint(db):
        return False, 'date mismatch'
    na, nb = extract_numbers(title_a), extract_numbers(title_b)
    if na and nb and not any(abs(a-b) < 0.5 for a in na for b in nb):
        return False, 'number mismatch'
    pa, pb = extract_proper_nouns(title_a), extract_proper_nouns(title_b)
    if pa and pb and pa.isdisjoint(pb):
        return False, 'proper-noun mismatch'
    return True, 'compatible'

print('Compatibility filters ready (date / number / proper-noun, regex-only).')


In [ ]:
# 5. GPU matching: bge-m3 → bge-reranker-v2-m3 → bge-reranker-v2-gemma (3-stage)
def compute_pair_arb(ma, mb):
    p1a = ma.get('bestBid', ma['yesPrice']); p1b = 1 - mb.get('bestAsk', mb['yesPrice'])
    cost1 = p1a + p1b
    roi1 = ((1 - cost1) / max(0.01, cost1) * 100) if 0.05 < cost1 < 1 else -100
    p2a = 1 - ma.get('bestAsk', ma['yesPrice']); p2b = mb.get('bestBid', mb['yesPrice'])
    cost2 = p2a + p2b
    roi2 = ((1 - cost2) / max(0.01, cost2) * 100) if 0.05 < cost2 < 1 else -100
    if roi1 >= roi2:
        return {'roi': min(1000, roi1), 'cost': cost1, 'scenario': 1}
    return {'roi': min(1000, roi2), 'cost': cost2, 'scenario': 2}

def _fuzzy_score(a: str, b: str) -> float:
    """rapidfuzz token sort ratio — 0-100."""
    return _rfuzz.token_sort_ratio(a, b)

def _entity_compatible(title_a: str, title_b: str) -> bool:
    """Return False only when both titles have named entities and they share none."""
    ea, eb = get_entities(title_a), get_entities(title_b)
    if ea and eb and ea.isdisjoint(eb):
        return False
    return True

def match_markets(markets, top_k=2000, min_similarity=0.62, min_roi=0.1,
                  candidate_top_k=50, min_fuzzy=20.0):
    t0 = time.time()
    by_platform = {}
    for m in markets:
        by_platform.setdefault(m['platform'], []).append(m)
    platforms = list(by_platform.keys())
    print(f'Platforms: {platforms}')

    # ── Stage 1: bge-m3 bi-encoder (cosine similarity matrix) ────────────────
    plat_embeddings = {}
    for plat in platforms:
        titles = [m['title'] for m in by_platform[plat]]
        print(f'  Encoding {len(titles):,} markets for {plat}...')
        plat_embeddings[plat] = encode_titles(titles)

    candidates = []
    for i in range(len(platforms)):
        pa = platforms[i]; emb_a = plat_embeddings[pa]; mkts_a = by_platform[pa]
        for j in range(i+1, len(platforms)):
            pb = platforms[j]; emb_b = plat_embeddings[pb]; mkts_b = by_platform[pb]
            print(f'  cos_sim {pa} x {pb} ({emb_a.shape[0]:,} x {emb_b.shape[0]:,})')
            scores = util.cos_sim(emb_a, emb_b)
            top    = torch.topk(scores, k=min(candidate_top_k, scores.shape[1]), dim=1)
            pair_count = 0
            for ai in range(top.values.shape[0]):
                ma = mkts_a[ai]
                for s_val, bi in zip(top.values[ai].cpu().tolist(), top.indices[ai].cpu().tolist()):
                    if s_val < min_similarity: break
                    mb = mkts_b[int(bi)]
                    ok, reason = are_compatible(ma['title'], mb['title'])
                    if not ok: continue
                    if not _entity_compatible(ma['title'], mb['title']): continue
                    arb = compute_pair_arb(ma, mb)
                    if arb['roi'] < min_roi: continue
                    fuzzy = _fuzzy_score(ma['title'], mb['title'])
                    candidates.append((ma, mb, arb['roi'], float(s_val), reason, fuzzy))
                    pair_count += 1
            print(f'    {pair_count:,} candidates')

    if not candidates:
        print('No candidates.'); return []

    # Dedup + sort by composite (bi_score + fuzzy_bonus)
    seen, deduped = set(), []
    for c in sorted(candidates, key=lambda x: x[3] + x[5]/200.0, reverse=True):
        key = tuple(sorted([c[0].get('id',''), c[1].get('id','')]))
        if key in seen: continue
        seen.add(key); deduped.append(c)

    # ── Stage 2: bge-reranker-v2-m3 (fast cross-encoder) ─────────────────────
    stage2_pool = deduped[:top_k * 4]
    print(f'Stage-2 reranking {len(stage2_pool):,} candidates (bge-reranker-v2-m3)...')
    pairs_s2 = [[c[0]['title'], c[1]['title']] for c in stage2_pool]
    s2_scores = reranker.predict(pairs_s2, show_progress_bar=True, batch_size=64)

    stage2_out = []
    for score, cand in zip(s2_scores, stage2_pool):
        if float(score) <= 0.3: continue
        ma, mb, roi, bi_score, reason, fuzzy = cand
        stage2_out.append({
            'ma': ma, 'mb': mb, 'roi': roi,
            'bi_score': bi_score, 's2_score': float(score), 'fuzzy': fuzzy,
        })
    stage2_out.sort(key=lambda x: x['s2_score'] + x['fuzzy']/200.0, reverse=True)
    print(f'Stage-2 survivors: {len(stage2_out):,}')

    # ── Stage 3: bge-reranker-v2-gemma (LLM-based, top-500 only) ─────────────
    stage3_input = stage2_out[:500]
    if deep_reranker and stage3_input:
        print(f'Stage-3 deep-reranking {len(stage3_input)} candidates (bge-reranker-v2-gemma)...')
        pairs_s3 = [[c['ma']['title'], c['mb']['title']] for c in stage3_input]
        s3_scores = deep_reranker.predict(pairs_s3, show_progress_bar=True, batch_size=8)
        for i, s3 in enumerate(s3_scores):
            stage3_input[i]['s3_score'] = float(s3)
        stage3_input.sort(key=lambda x: x.get('s3_score', 0), reverse=True)
        # Merge: stage-3 scored items first, then remainder from stage-2
        s3_ids = {tuple(sorted([c['ma'].get('id',''), c['mb'].get('id','')])) for c in stage3_input}
        stage2_remainder = [c for c in stage2_out[500:]
                             if tuple(sorted([c['ma'].get('id',''), c['mb'].get('id','')])) not in s3_ids]
        merged = stage3_input + stage2_remainder
    else:
        merged = stage2_out

    final = []
    for c in merged[:top_k]:
        ma, mb = c['ma'], c['mb']
        s3 = c.get('s3_score', c['s2_score'])
        final.append({
            'marketA':       {'id': ma.get('id'), 'platform': ma['platform'], 'title': ma['title'],
                              'marketUrl': ma.get('marketUrl',''), 'yesPrice': ma['yesPrice'],
                              'noPrice': ma.get('noPrice', round(1-ma['yesPrice'],4)),
                              'endDate': ma.get('endDate')},
            'marketB':       {'id': mb.get('id'), 'platform': mb['platform'], 'title': mb['title'],
                              'marketUrl': mb.get('marketUrl',''), 'yesPrice': mb['yesPrice'],
                              'noPrice': mb.get('noPrice', round(1-mb['yesPrice'],4)),
                              'endDate': mb.get('endDate')},
            'roi':           c['roi'],
            'matchScore':    round(s3 * 100, 1),
            'biEncoderScore':round(c['bi_score'], 4),
            'fuzzyScore':    round(c['fuzzy'], 1),
            'stage2Score':   round(c['s2_score'], 4),
            'stage3Score':   round(c.get('s3_score', c['s2_score']), 4),
            'matchReason':   c.get('reason', 'compatible'),
            'isVerified':    s3 > 0.80,
        })

    final.sort(key=lambda x: (x['isVerified'], x['matchScore'], x['roi']), reverse=True)
    print(f'Done in {time.time()-t0:.1f}s. {len(final):,} pass all 3 stages.')
    return final

found_pairs = match_markets(all_markets, top_k=2000, min_similarity=0.62, min_roi=0.1)
print(f'\nFinal: {len(found_pairs):,} arbitrage pairs.')


In [ ]:
# 6. Send results back over the same WebSocket tunnel
async def send_results_via_websocket(pairs):
    if not pairs:
        print('No pairs to send.'); return False
    print(f'Sending {len(pairs):,} pairs back to backend...')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({
            'type': 'cloud_results',
            'data': pairs,
            'count': len(pairs),
            'timestamp': time.time(),
        }))
        ack = json.loads(await ws.recv())
        if ack.get('type') == 'results_received':
            print(f'Backend ack: {ack.get("message","")}')
            return True
        print(f'Unexpected ack: {ack}'); return False

if found_pairs:
    ok = asyncio.get_event_loop().run_until_complete(send_results_via_websocket(found_pairs))
    if ok:
        print('Pipeline complete - results returned through the ngrok tunnel.')
else:
    print('No pairs to send.')


In [ ]:
# 7. Preview top opportunities
print('=' * 90)
print(f'TOP {min(10, len(found_pairs))} ARBITRAGE OPPORTUNITIES')
print('=' * 90)
for i, p in enumerate(found_pairs[:10], 1):
    badge = 'VERIFIED' if p['isVerified'] else 'review'
    print(f"\n#{i}  ROI: {p['roi']:.2f}%  match: {p['matchScore']}%  ({badge})")
    print(f"  [A] {p['marketA']['platform']:12}  {p['marketA']['title']}")
    print(f"        YES {p['marketA']['yesPrice']:.3f}  NO {p['marketA']['noPrice']:.3f}")
    print(f"  [B] {p['marketB']['platform']:12}  {p['marketB']['title']}")
    print(f"        YES {p['marketB']['yesPrice']:.3f}  NO {p['marketB']['noPrice']:.3f}")
